In [17]:
import numpy as np
from kinisi.parser import _get_matrix

In [18]:
def get_disp(coords, latt):
   """
   Calculate displacements for NVT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array of with shape [site, time step, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1) #change array shape and removes extra dim
   d_coords = coords[:, 1:] - coords[:, :-1] #calculate difference in frac, could use np.diff()
   d_coords = d_coords - np.round(d_coords) #when difference is greater than 0.5 -> wrap difference; -1 when greater than 0.5
   f_disp = np.cumsum(d_coords, axis=1) #sum over ?
   latt = np.array(latt) #convert latt from linked list to array
   disp = np.einsum('ijk,jkl->ijk', f_disp, latt[1:]) #apply lattice vectors to displacements
   return disp


def get_disp_npt(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1) #change array shape and removes extra dim
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt) #apply lattice vectors to get cartisian coords
   wrapped_diff = np.diff(wrapped, axis=1) #calculate difference in cart
   latt_para = np.einsum('ijj->ij', latt) #get lattice lengths; only works for lattice angles = 90°

   unwrapped_disp = wrapped_diff - (np.floor(wrapped_diff / latt_para[1:] + 1 / 2) * latt_para[1:]) #apply TOR scheme

   return np.cumsum(unwrapped_disp, axis=1)


def get_disp_npt_matrix(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1) #change array shape and removes extra dim
   latt_inv = np.linalg.inv(latt) #invert lattice vectors
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt) #apply lattice vectors to get cartisian coords
   wrapped_diff = np.diff(wrapped, axis=1) #calculate difference in cart

   unwrapped_disp = wrapped_diff - (np.einsum('ijk,jkl->ijk', np.floor(np.einsum('ijk,jkl->ijk', wrapped_diff, latt_inv[1:]) + (1 / 2)), latt[1:])) #should split this up for readability

   return np.cumsum(unwrapped_disp, axis=1)

HLAT: $$ u_{i+1} = w_{i+1} - \left\lfloor \frac{(w_{i+1} - u_i)}{L_{i+1}} \right\rfloor$$

TOR: $$ u_{i+1} = u_i + (w_{i+1} - w_i) - \left\lfloor \frac{w_{i+1} - w_i}{L_{i+1}}\right\rfloor L_{i+1} $$

In [19]:
coords = np.array([[[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.2,0.2,0.2],[0.2,0.2,0.2],[0.2,0.2,0.2],[0.2,0.2,0.2]],
                   [[0.3,0.2,0.2],[0.3,0.2,0.2],[0.3,0.2,0.2],[0.3,0.2,0.2]],
                   [[0.4,0.2,0.2],[0.4,0.2,0.2],[0.4,0.2,0.2],[0.4,0.2,0.2]]
                   ])
coords = np.expand_dims(coords, 2)

latt = np.array([[[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                ])

nvt_disp = get_disp(coords, latt)
npt_disp = get_disp_npt(coords, latt)
npt_matrix_disp = get_disp_npt_matrix(coords, latt)

np.testing.assert_array_almost_equal(npt_matrix_disp, nvt_disp)

In [20]:
print(get_disp(coords, latt))

[[[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]]


In [21]:
print(get_disp_npt(coords, latt))

[[[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]]


In [22]:
print(get_disp_npt_matrix(coords, latt))

[[[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]

 [[0. 0. 0.]
  [1. 1. 1.]
  [2. 1. 1.]
  [3. 1. 1.]]]


In [28]:
matrices = []

with open('lattice_params.csv') as infile:
   lines = infile.readlines()
   for line in lines[1:]:
      line = line.strip('\n')
      params = [float(x) for x in line.split(',')]
      if np.all(np.array(params)[[-1,-2,-3]] == 90):
         matrices.append(_get_matrix(params))

print(np.array(matrices).shape)

(10280, 3, 3)


In [24]:
for i, matrix in enumerate(matrices):
   matrices[i] = np.linalg.inv(matrix)
print(matrices)

[array([[ 1.78699071e-01,  0.00000000e+00, -1.60883710e-17],
       [-2.87370023e-17,  9.71817298e-02, -1.60883710e-17],
       [ 0.00000000e+00,  0.00000000e+00,  2.62743037e-01]]), array([[ 1.87476565e-01,  0.00000000e+00, -8.50921900e-18],
       [-3.01485311e-17,  1.58528852e-01, -8.50921900e-18],
       [ 0.00000000e+00,  0.00000000e+00,  1.38966092e-01]]), array([[ 1.27551020e-01,  0.00000000e+00, -9.81759499e-18],
       [-2.05117685e-17,  6.92952671e-02, -9.81759499e-18],
       [ 0.00000000e+00,  0.00000000e+00,  1.60333494e-01]]), array([[ 1.26230750e-01,  0.00000000e+00, -9.80815953e-18],
       [-2.02994528e-17,  1.38140627e-01, -9.80815953e-18],
       [ 0.00000000e+00,  0.00000000e+00,  1.60179401e-01]]), array([[ 9.96710854e-02,  0.00000000e+00, -6.10309379e-18],
       [-1.60283330e-17,  9.96710854e-02, -6.10309379e-18],
       [ 0.00000000e+00,  0.00000000e+00,  9.96710854e-02]]), array([[ 1.83351669e-01,  0.00000000e+00, -2.85319137e-18],
       [-2.94851971e-17,  1.7

In [25]:
random_coords = np.random.rand(*coords.shape)
print(random_coords.shape)
print(random_coords)

random_coords = np.array(
    [[[[0.14973271, 0.11130601, 0.77991697]],  
      [[0.8830278 , 0.82101692, 0.38448396]],
      [[0.97841794, 0.51656344, 0.11532697]],
      [[0.76991731, 0.67949457, 0.43153843]]],
     [[[0.84796091, 0.78852347, 0.63514311]],
      [[0.94440032, 0.75935275, 0.32524654]],
      [[0.3411861 , 0.83430718, 0.49755573]],
      [[0.01221086, 0.09876532, 0.34103097]]],
     [[[0.27181602, 0.77118809, 0.0316385 ]],
      [[0.30708704, 0.09420535, 0.33950568]],
      [[0.23770583, 0.55500863, 0.4509913 ]],
      [[0.24460106, 0.38571214, 0.72515897]]],
     [[[0.15985577, 0.06524875, 0.08743626]],
      [[0.90965231, 0.73967972, 0.24293762]],
      [[0.13836285, 0.42984764, 0.39586935]],
      [[0.33348705, 0.64147203, 0.69719941]]],
     [[[0.00195463, 0.78259409, 0.94402232]],
      [[0.84560141, 0.49460504, 0.91815449]],
      [[0.73329605, 0.66132599, 0.57232998]],
      [[0.84946125, 0.87500532, 0.06945798]]]]
)

(5, 4, 1, 3)
[[[[0.96002583 0.41725989 0.07925514]]

  [[0.84978887 0.55875815 0.95864504]]

  [[0.37181897 0.62143813 0.21186814]]

  [[0.91812775 0.78842728 0.98742269]]]


 [[[0.4547165  0.74553211 0.7089716 ]]

  [[0.99719055 0.80508621 0.43658172]]

  [[0.55107999 0.24983869 0.87901615]]

  [[0.0565856  0.63225993 0.68711803]]]


 [[[0.48123848 0.99617625 0.49345838]]

  [[0.00463102 0.51401644 0.13397804]]

  [[0.49834326 0.15809162 0.5086609 ]]

  [[0.6581716  0.82089681 0.75839438]]]


 [[[0.27291357 0.80296471 0.82430019]]

  [[0.59320169 0.26155733 0.47294838]]

  [[0.39428569 0.17276454 0.71176881]]

  [[0.31670797 0.54758924 0.9006625 ]]]


 [[[0.49820375 0.75194186 0.25488211]]

  [[0.51768693 0.81093972 0.84331015]]

  [[0.32627635 0.11894518 0.45177892]]

  [[0.71926523 0.54451685 0.90782196]]]]


In [26]:
for matrix in matrices:
   latt = np.tile(matrix, (5,1,1))
   nvt = get_disp(random_coords, latt)
   npt = get_disp_npt_matrix(random_coords, latt)
   try:
      np.testing.assert_almost_equal(nvt, npt)
   except:
      difference = nvt - npt
      difference[difference < 1e-7] = 0
      print(matrix)
      print(difference)
      np.testing.assert_almost_equal(nvt, npt)

In [27]:
zero_matrices = np.array(matrices)
zero_matrices[zero_matrices < 1e-7] = 0

for matrix in zero_matrices:
   latt = np.tile(matrix, (5,1,1))
   nvt = get_disp(random_coords, latt)
   npt = get_disp_npt_matrix(random_coords, latt)
   try:
      np.testing.assert_almost_equal(nvt, npt)
   except:
      difference = nvt - npt
      difference[difference < 1e-7] = 0
      print('matrix\n', matrix)
      print('difference\n', difference)
      np.testing.assert_almost_equal(nvt, npt)

$$
X = 
\left(\begin{array}{cc}
a & 0 & c\\
j & k & l\\
0 & 0 & z
\end{array}\right)
$$

$$
X^{-1} = 
\left(\begin{array}{cc}
1/a & 0 & -c/az\\
-j/ak & 1/k & cj-al/akz\\
0 & 0 & 1/z
\end{array}\right)
$$
